# 11 — Claude Haiku 4.5 (Anthropic su Azure AI Foundry)

Valuta **Claude Haiku 4.5** servito da Azure AI Foundry (catalogo modelli Anthropic), con lo
stesso prompt zero-shot e gli stessi parametri dei notebook 07–10. Il client è `AnthropicFoundry`
dell'SDK `anthropic` (l'endpoint Foundry usa l'API Messages di Anthropic, non chat-completions).

Ambiti come nel notebook 10: `sample` (centesimi) e `test` (5.610 righe, ~25 $ a 1 $/M input e
5 $/M output — Claude su Foundry **non** ha una Batch API, quindi la split test è il massimo
ambito ragionevole per questo modello).

Prerequisiti: modello Claude Haiku 4.5 attivato nel catalogo Foundry della risorsa e variabili
d'ambiente `AZURE_ANTHROPIC_API_KEY` e `AZURE_ANTHROPIC_RESOURCE` (nome della risorsa Foundry).

## Ripresa e rischio di mescolare corse

⚠️ Il ciclo condiviso salta i `row_id` già presenti nel TSV di output: rieseguire la generazione
su un file esistente **aggiunge solo le righe mancanti**. È il comportamento voluto per riprendere
una corsa interrotta, ma con un modello, un deployment o una configurazione diversi si
mescolerebbero due corse nello stesso file: in quel caso **eliminare prima** il TSV e rigenerare
tutto (rieseguendo poi la valutazione). Ogni ambito (`sample`, `test`) scrive su un file separato.

In [ ]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer
try:
    import anthropic  # noqa: F401
except ImportError:
    %pip install anthropic

In [ ]:
# --- Configurazione ---------------------------------------------------------
import os
import summ_utils as su

METODO     = 'haiku'
SCOPE      = 'sample'    # 'sample' = campione condiviso; 'test' = intera split test (5.610 righe)
N_SAMPLES  = 100
SEED       = 42
LIMIT      = None        # es. 3 per uno smoke test rapido; None = tutti

MODELLO  = 'claude-haiku-4-5'
RESOURCE = os.environ['AZURE_ANTHROPIC_RESOURCE']   # nome della risorsa Azure AI Foundry
API_KEY  = os.environ['AZURE_ANTHROPIC_API_KEY']

MAX_TOKENS  = 300
TEMPERATURE = 0.3
PROMPT_SYSTEM = ('You are a helpful assistant that summarizes news articles '
                 'from different sources concisely.')
PROMPT_USER   = 'Summarize the following document into a comprehensive summary: {documento}'
ETICHETTA   = 'Haiku '
NOTE_CONFIG = ('prompt zero-shot in inglese identico ai notebook 07-10; API Messages di '
               'Anthropic via AnthropicFoundry; ambiti sample e test')

BASE = su.trova_base_dir()
P    = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH    = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'

config = {'modello': MODELLO,
          'backend': 'Azure AI Foundry (AnthropicFoundry, API Messages)',
          'max_tokens': MAX_TOKENS, 'temperature': TEMPERATURE,
          'prompt_system': PROMPT_SYSTEM, 'prompt_user': PROMPT_USER,
          'note': NOTE_CONFIG}

print(f'Modello : {MODELLO} sulla risorsa Foundry {RESOURCE!r}')
print(f'Ambito  : {SCOPE}')
print(f'Output  : {OUT_PATH}')

## Generazione dei riassunti

L'API Messages restituisce una lista di blocchi di contenuto: si concatenano i soli blocchi
`text`. Una risposta senza testo solleva un'eccezione, così la riga viene registrata come errore
e **non** scritta nel TSV (ritentabile alla corsa successiva).

In [ ]:
from anthropic import AnthropicFoundry

client = AnthropicFoundry(api_key=API_KEY, resource=RESOURCE)

def genera(documento):
    risposta = client.messages.create(
        model=MODELLO,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        system=PROMPT_SYSTEM,
        messages=[{'role': 'user', 'content': PROMPT_USER.format(documento=documento)}])
    testo = ''.join(b.text for b in risposta.content if b.type == 'text').strip()
    if not testo:
        # solleva -> il ciclo condiviso registra l'errore e NON scrive la riga
        raise RuntimeError(f'risposta vuota (stop_reason={risposta.stop_reason})')
    return testo

if SCOPE == 'sample':
    esempi = su.carica_campione(SAMPLE_PATH)
elif SCOPE == 'test':
    esempi = su.itera_split(P['complete_tab'], 'test')
else:
    raise ValueError(f'SCOPE non valido: {SCOPE!r}')

scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(esempi, scrittore, genera, limit=LIMIT,
                                etichetta=ETICHETTA)
scrittore.chiudi()

## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare i riassunti. Metriche ROUGE-1/2/L
(F1, precisione, recall), BLEU e METEOR con normalizzazione identica per tutti i metodi del
benchmark. Output: `results/metrics/{metodo}_{scope}_per_example.csv` e `…_aggregate.json`.
Per l'ambito `test` i riferimenti vengono letti in streaming da `complete.tab`.

In [ ]:
import json

riassunti = su.carica_riassunti(OUT_PATH)
if SCOPE == 'sample':
    riferimenti = su.carica_campione(SAMPLE_PATH)
else:
    riferimenti = su.itera_split(P['complete_tab'], 'test')

righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                     P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))
print('\nMedie per split:')
for split, valori in aggregato['per_split'].items():
    print(f"  {split:5s} (n={valori['n_esempi']}): ROUGE-1 F1 = {valori['rouge1_f1']:.3f}")

## Ispezione qualitativa

In [ ]:
if SCOPE == 'sample':
    riferimenti = su.carica_campione(SAMPLE_PATH)
else:
    riferimenti = su.itera_split(P['complete_tab'], 'test')
su.mostra_esempi(riferimenti, riassunti, quanti=2)